# Part B: Collection Plan with Optimization (Pyomo)
AIPI 530 · Fall 2026 · Oklahoma Mothers' Milk Bank

One truck leaves the milk bank each day (Monday to Friday, at most 5 routes), picks up at depots, and returns to the bank within 720 minutes.
Every depot is served exactly once: either the truck picks up there, or the depot ships directly to the bank.
Choose the routes and the shipping depots that **minimize total weekly cost** (labor + fuel + shipping).

This is the class routing model (arc variables, one arrival and one departure per stop, depot departures and returns, ordering labels),
with driving time in place of load, and a binary variable that lets a depot ship.

**Packages:** `pandas`, `openpyxl`, `pyomo`, `highspy`. **Input files** (`data` folder): `Depot Information.xlsx`, `Optional - Driving Distance and Time.xlsx`.
Open this notebook from the `code` folder and run the cells top to bottom.

In [1]:
# Define packages
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

In [2]:
# DISTANCE and TIME data (one row for each ordered pair; Time is in minutes and Distance in miles)
depotData = pd.read_excel('../data/Depot Information.xlsx', sheet_name='Depots')
travelData = pd.read_excel('../data/Optional - Driving Distance and Time.xlsx')
travelData.columns = ['Pair', 'From', 'To', 'Time', 'Distance']

BANK = 'Oklahoma Mothers’ Milk Bank'
DEPOTS = [d for d in depotData['Depot Designation'] if d != BANK]   # the 36 depots
LOCATIONS = [BANK] + DEPOTS

print(len(DEPOTS), 'depots,', len(travelData), 'directed driving-time pairs')
travelData.head()

36 depots, 1332 directed driving-time pairs


,Pair,From,To,Time,Distance
0,Oklahoma Mothers’ Milk Bank|Integris Southwest,Oklahoma Mothers’ Milk Bank,Integris Southwest,12.416667,6.7437
1,Oklahoma Mothers’ Milk Bank|Integris Baptist,Oklahoma Mothers’ Milk Bank,Integris Baptist,14.050000,8.8166
2,Oklahoma Mothers’ Milk Bank|Cleveland County M...,Oklahoma Mothers’ Milk Bank,Cleveland County Moore,19.383333,11.4450
3,Oklahoma Mothers’ Milk Bank|Variety Care- Lafa...,Oklahoma Mothers’ Milk Bank,Variety Care- Lafayette,13.200000,7.4378
4,Oklahoma Mothers’ Milk Bank|Edmond Hope Center,Oklahoma Mothers’ Milk Bank,Edmond Hope Center,20.933333,15.1602


In [3]:
# CASE PARAMETERS
paramData = pd.DataFrame({'fuelPrice': [2.70], 'mpg': [21], 'laborRate': [7.25], 'shipRate': [1.00],
                          'maxMinutes': [720], 'maxRoutes': [5]})
paramData

,fuelPrice,mpg,laborRate,shipRate,maxMinutes,maxRoutes
0,2.7,21,7.25,1.0,720,5


In [4]:
# Create empty model
m = AbstractModel()

**Sets and parameters**

- $V$: set of locations (the bank and the depots)
- $C = V \setminus \{Bank\}$: set of depots
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs
- $t_{ij}$: driving time from $i$ to $j$ (minutes)
- $d_{ij}$: driving distance from $i$ to $j$ (miles)
- $T$: maximum minutes per route (720), $K$: maximum number of routes (5)
- fuelPrice ($ per gallon), mpg (miles per gallon), laborRate ($ per hour), shipRate ($ per mile)

In [5]:
# Define Sets
m.V = Set()
m.C = Set(within=m.V, initialize=lambda m: [i for i in m.V if i != BANK])
m.A = Set(within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j])

# Define Params
m.time = Param(m.A)
m.dist = Param(m.A)
m.fuelPrice = Param()
m.mpg = Param()
m.laborRate = Param()
m.shipRate = Param()
m.maxMinutes = Param()
m.maxRoutes = Param()

**Decision variables**

- $x_{ij} \in \{0,1\}$: 1 if the truck drives directly from $i$ to $j$
- $s_i \in \{0,1\}$: 1 if depot $i$ ships directly to the bank
- $u_i$: minutes driven since leaving the bank when the truck arrives at depot $i$ (the class load label, but time)
- Implicit variables: LaborCost, FuelCost, ShipCost, TotalCost, DrivingMinutes, TruckMiles

$t_{Bank,i} \le u_i \le T - t_{i,Bank} \qquad \forall i \in C$
(the arrival time is at least the drive from the bank, and still leaves time to drive back)

In [6]:
# Define Decision Vars
m.x = Var(m.A, domain=Binary)
m.s = Var(m.C, domain=Binary)
m.u = Var(m.C, domain=NonNegativeReals, bounds=lambda m, i: (m.time[BANK, i], m.maxMinutes - m.time[i, BANK]))

# Define Implicit Vars
m.LaborCost = Var(domain=NonNegativeReals)
m.FuelCost = Var(domain=NonNegativeReals)
m.ShipCost = Var(domain=NonNegativeReals)
m.TotalCost = Var(domain=NonNegativeReals)
m.DrivingMinutes = Var(domain=NonNegativeReals)
m.TruckMiles = Var(domain=NonNegativeReals)

**Enter each depot exactly once (or it ships)**

$\sum_{i \in V,\, i \ne j} x_{ij} + s_j = 1 \qquad \forall j \in C$

In [7]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) + m.s[j] == 1
m.EnterOnce = Constraint(m.C, rule=Enter_Once_Rule)

**Leave each depot exactly once (or it ships)**

$\sum_{j \in V,\, j \ne i} x_{ij} + s_i = 1 \qquad \forall i \in C$

In [8]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) + m.s[i] == 1
m.LeaveOnce = Constraint(m.C, rule=Leave_Once_Rule)

**Departures from the bank**

At most one route per weekday.

$\sum_{j \in C} x_{Bank,\,j} \le K$

In [9]:
def Bank_Departures_Rule(m):
    return sum(m.x[BANK, j] for j in m.C) <= m.maxRoutes
m.BankDepartures = Constraint(rule=Bank_Departures_Rule)

**Returns to the bank**

Every route that leaves the bank returns to it.

$\sum_{i \in C} x_{i,\,Bank} = \sum_{j \in C} x_{Bank,\,j}$

In [10]:
def Bank_Returns_Rule(m):
    return sum(m.x[i, BANK] for i in m.C) == sum(m.x[BANK, j] for j in m.C)
m.BankReturns = Constraint(rule=Bank_Returns_Rule)

**Time propagation (720-minute limit and subtour elimination)**

If the truck goes from depot $i$ to depot $j$, then $u_j \ge u_i + t_{ij}$. Otherwise the constraint is not binding.
This is the class load constraint with driving time in place of load (and $T + t_{ij}$ in place of $Q$).

$u_i - u_j + (T + t_{ij})\, x_{ij} \le T \qquad \forall (i,j) \in A,\; i, j \ne Bank$

In [11]:
def Time_Rule(m, i, j):
    if i == BANK or j == BANK:
        return Constraint.Skip
    return m.u[i] - m.u[j] + (m.maxMinutes + m.time[i, j]) * m.x[i, j] <= m.maxMinutes
m.Time = Constraint(m.A, rule=Time_Rule)

**Cost definitions**

DrivingMinutes $= \sum t_{ij} x_{ij}$, TruckMiles $= \sum d_{ij} x_{ij}$,
LaborCost = (laborRate / 60) · DrivingMinutes, FuelCost = (fuelPrice / mpg) · TruckMiles,
ShipCost $= \sum_i$ shipRate · $d_{i,Bank}$ · $s_i$, TotalCost = LaborCost + FuelCost + ShipCost

In [12]:
def Define_Minutes_Rule(m):
    return m.DrivingMinutes == sum(m.time[i, j] * m.x[i, j] for (i, j) in m.A)
m.Define_Minutes = Constraint(rule=Define_Minutes_Rule)

def Define_Miles_Rule(m):
    return m.TruckMiles == sum(m.dist[i, j] * m.x[i, j] for (i, j) in m.A)
m.Define_Miles = Constraint(rule=Define_Miles_Rule)

def Define_Labor_Rule(m):
    return m.LaborCost == (m.laborRate / 60) * m.DrivingMinutes
m.Define_Labor = Constraint(rule=Define_Labor_Rule)

def Define_Fuel_Rule(m):
    return m.FuelCost == (m.fuelPrice / m.mpg) * m.TruckMiles
m.Define_Fuel = Constraint(rule=Define_Fuel_Rule)

def Define_Ship_Rule(m):
    return m.ShipCost == sum(m.shipRate * m.dist[i, BANK] * m.s[i] for i in m.C)
m.Define_Ship = Constraint(rule=Define_Ship_Rule)

def Define_Total_Rule(m):
    return m.TotalCost == m.LaborCost + m.FuelCost + m.ShipCost
m.Define_Total = Constraint(rule=Define_Total_Rule)

**Minimize total weekly cost**

$\min \; TotalCost$

In [13]:
def Min_Cost_Rule(m):
    return m.TotalCost
m.Min_Cost = Objective(rule=Min_Cost_Rule, sense=minimize)

In [14]:
# Build and read instance data
instanceData = {None: {
    'V': {None: LOCATIONS},
    'time': travelData.set_index(['From', 'To'])['Time'].to_dict(),
    'dist': travelData.set_index(['From', 'To'])['Distance'].to_dict(),
    'fuelPrice': {None: paramData.loc[0, 'fuelPrice']},
    'mpg': {None: paramData.loc[0, 'mpg']},
    'laborRate': {None: paramData.loc[0, 'laborRate']},
    'shipRate': {None: paramData.loc[0, 'shipRate']},
    'maxMinutes': {None: paramData.loc[0, 'maxMinutes']},
    'maxRoutes': {None: paramData.loc[0, 'maxRoutes']},
}}
instance = m.create_instance(instanceData)

**Solve**

About 1,300 binary variables, so proving optimality can take a very long time. We set a time limit and report the solver status and the gap.
A plan returned at the time limit is **feasible**; it is **proven optimal** only if the gap is 0.
(GLPK, the class solver, is too slow for a problem this size, so we use HiGHS through `appsi_highs`.)

In [15]:
TIME_LIMIT = 1800   # seconds (set to 300 for a quick test run)

solver = SolverFactory('appsi_highs')
solver.config.stream_solver = True
solver.highs_options["time_limit"] = TIME_LIMIT
solver.config.time_limit = TIME_LIMIT
solver.config.mip_gap = 0.0
solution = solver.solve(instance)

load_solution=False and check results.termination_condition and
results.found_feasible_solution() before loading a solution.


In [16]:
best = value(instance.TotalCost)  # best plan found (feasible)
bound = solution.problem.lower_bound  # best lower bound proven by HiGHS
print("Termination condition:", solution.solver.termination_condition)
print("Best plan found (feasible) :", best)
print("Best lower bound           :", bound)
print("Gap                        :", f"{100 * (best - bound) / best:.2f}%")
print("Total weekly cost          :", value(instance.TotalCost))

Termination condition: maxTimeLimit
Best plan found (feasible) : 657.5531079365078
Best lower bound           : 578.1760939880953
Gap                        : 12.07%
Total weekly cost          : 657.5531079365078


**Create output data:** start one route at each departure from the bank and follow the arcs back to the bank

In [17]:
time = travelData.set_index(['From', 'To'])['Time'].to_dict()
dist = travelData.set_index(['From', 'To'])['Distance'].to_dict()
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

next_stop = {i: j for (i, j) in instance.A if i != BANK and value(instance.x[i, j]) > 0.5}
first_stops = [j for j in instance.C if value(instance.x[BANK, j]) > 0.5]
shipDepots = [i for i in instance.C if value(instance.s[i]) > 0.5]

routes = []
for first in first_stops:
    route = [BANK, first]
    while route[-1] != BANK:
        route.append(next_stop[route[-1]])
    routes.append(route)

dayRows, stopRows = [], []
for k, route in enumerate(routes):
    legs = list(zip(route[:-1], route[1:]))
    T = sum(time[a, b] for a, b in legs)
    D = sum(dist[a, b] for a, b in legs)
    dayRows.append({'Day': days[k], 'Depots visited': len(route) - 2, 'Driving minutes': T, 'Driving hours': T / 60, 'Miles': D})
    cum = 0.0
    for n, (a, b) in enumerate(legs, start=1):
        cum += time[a, b]
        stopRows.append({'Day': days[k], 'Leg': n, 'From': a, 'To': b, 'Minutes': time[a, b], 'Time since departure': cum, 'Miles': dist[a, b]})

dayTable = pd.DataFrame(dayRows)
stopTable = pd.DataFrame(stopRows)
print(dayTable.round(2).to_string(index=False))
print()
for k, route in enumerate(routes):
    print(f'{days[k]}: ' + ' → '.join(route))
    print()
print('Ship directly:', shipDepots if shipDepots else 'none')
print()
print(f'Total driving time : {value(instance.DrivingMinutes):,.1f} min ({value(instance.DrivingMinutes) / 60:.2f} h)')
print(f'Total truck miles  : {value(instance.TruckMiles):,.1f}')
print(f'Labor              : ${value(instance.LaborCost):,.2f}')
print(f'Fuel               : ${value(instance.FuelCost):,.2f}')
print(f'Direct shipping    : ${value(instance.ShipCost):,.2f}  ({len(shipDepots)} depots)')
print(f'TOTAL WEEKLY COST  : ${value(instance.TotalCost):,.2f}')

      Day  Depots visited  Driving minutes  Driving hours  Miles
   Monday              11           510.30           8.51 498.67
  Tuesday              11           642.75          10.71 653.07
Wednesday              10           711.98          11.87 748.56
 Thursday               4           687.63          11.46 814.97

Monday: Oklahoma Mothers’ Milk Bank → Integris Southwest → Variety Care- Lafayette → Grady County → Comanche County Memorial Hospital → Jackson County Altus → Beckham County → Custer County → Alliance Health Woodward → Garfield County Health Department → Canadian Valley → Integris Baptist → Oklahoma Mothers’ Milk Bank

Tuesday: Oklahoma Mothers’ Milk Bank → St. Anthony Shawnee Hospital → Tulsa Hillcrest Hospital → YWCA South → Washington Regional Medical Center → McIntosh County → McAlester Regional Health → Pontotoc County → Carter County Health Department → South Central Medical Center → Cleveland County Moore → OU Children’s Hospital → Oklahoma Mothers’ Milk Bank

**Independent feasibility check:** re-checks the case rules on the plan above, using the original data (not the model)

In [18]:
visited = [d for r in routes for d in r[1:-1]]
assert len(routes) <= 5, 'more than 5 routes'
assert all(r[0] == BANK and r[-1] == BANK for r in routes), 'every route must start and end at the bank'
assert sorted(visited + shipDepots) == sorted(DEPOTS), 'every depot must be served exactly once'
assert len(visited) == len(set(visited)), 'a depot is visited twice'
for k, r in enumerate(routes):
    assert sum(time[a, b] for a, b in zip(r[:-1], r[1:])) <= 720 + 1e-6, f'{days[k]} exceeds 720 minutes'
recost = (7.25 / 60) * dayTable['Driving minutes'].sum() + (2.70 / 21) * dayTable['Miles'].sum() + sum(dist[i, BANK] for i in shipDepots)
assert abs(recost - value(instance.TotalCost)) < 1e-4, 'cost from the plan does not match the model'
print('All operating rules satisfied; plan cost matches the model objective.')

All operating rules satisfied; plan cost matches the model objective.
